# AI Resume Screening and Candidate Ranking System
**How to run:** select your conda kernel (top right), click **Restart**, then **Run All**. Cells must run in order, top to bottom.

**Structure (3 modules, written by this notebook):**
- `extractor.py`: resume extraction (cells in section 5)
- `matcher.py`: JD keywords + explainable scoring (section 6)
- `app.py`: Streamlit web interface (section 7)

The notebook also has its own in-notebook dashboard (section 10). For the web app, run `streamlit run app.py` in a terminal.

## 1. Install (first time only)

In [ ]:
!pip install pdfplumber python-docx scikit-learn pandas ipywidgets streamlit "reportlab==4.2.5"

## 2. Imports and folders

In [ ]:
import os, re
import pandas as pd
import docx
import ipywidgets as widgets
from IPython.display import display, clear_output

os.makedirs("sample_data/resumes", exist_ok=True)
os.makedirs("sample_data/job_descriptions", exist_ok=True)
print("setup done")

## 3. Fake job descriptions

In [ ]:
jd_data = """Data Scientist - Job Description
We are looking for a Data Scientist with strong skills in Python, SQL, Pandas, NumPy and Machine Learning.
Experience with Scikit-learn, TensorFlow, Data Visualization, Statistics and NLP is required.
Knowledge of Power BI or Tableau, Git and Jupyter Notebook is a plus.
Candidates should have a Bachelor's degree in Computer Science or related field and 1+ years of experience.
Strong communication and problem solving skills are expected."""

jd_web = """Full Stack Web Developer - Job Description
We are hiring a Full Stack Developer skilled in JavaScript, React, Node.js, HTML, CSS and REST APIs.
Experience with MongoDB, SQL, Git, Docker and AWS is required.
Knowledge of TypeScript, Express, Agile and CI/CD is a plus.
Candidates should have a Bachelor's degree in Computer Science or related field and 1+ years of experience.
Good teamwork and problem solving skills are expected."""

with open("sample_data/job_descriptions/jd_data_scientist.txt", "w") as f: f.write(jd_data)
with open("sample_data/job_descriptions/jd_web_developer.txt", "w") as f: f.write(jd_web)
print("2 JDs saved")

## 4. Fake resumes (10 DOCX + 5 PDF, all fictional)

In [ ]:
import random
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer

random.seed(42)   # same fake data every run -> consistent test results

first = ["Aarav","Meera","Rohan","Sneha","Kabir","Ananya","Vikram","Isha","Arjun","Neha","Rahul","Priya","Karan","Divya","Sahil"]
last  = ["Sharma","Iyer","Nair","Rao","Mehta","Patel","Singh","Das","Kulkarni","Reddy","Gupta","Joshi","Bose","Menon","Verma"]

DATA_SKILLS = ["Python","SQL","Pandas","NumPy","Machine Learning","Scikit-learn","TensorFlow",
               "Data Visualization","Statistics","NLP","Power BI","Tableau","Git","Jupyter Notebook","Deep Learning"]
WEB_SKILLS  = ["JavaScript","React","Node.js","HTML","CSS","REST APIs","MongoDB","SQL","Git",
               "Docker","AWS","TypeScript","Express","Agile","CI/CD"]
COMMON      = ["Communication","Problem Solving","Teamwork"]
CERTS       = ["AWS Cloud Practitioner","Google Data Analytics Certificate","Meta Front-End Developer Certificate",
               "Microsoft Azure Fundamentals","Coursera Machine Learning Specialization"]
DEGREES     = ["B.E. Computer Science","B.Tech Information Technology","B.Sc Computer Science","BCA"]

def build_candidate(i):
    # even index -> data profile, odd index -> web profile
    profile = "data" if i % 2 == 0 else "web"
    pool = DATA_SKILLS if profile == "data" else WEB_SKILLS
    skills = random.sample(pool, random.randint(6, 11)) + random.sample(COMMON, 2)
    years = random.randint(0, 4)
    role = "Data Analyst Intern" if profile == "data" else "Web Developer Intern"
    return {
        "name": f"{first[i]} {last[i]}",
        "email": f"{first[i].lower()}.{last[i].lower()}@example.com",
        "skills": skills,
        "education": f"{random.choice(DEGREES)}, Sample Institute of Technology, 2024",
        "experience": f"{role} - {years} years. Worked on {', '.join(skills[:3])} projects.",
        "certs": [] if i == 13 else random.sample(CERTS, 1),   # resume 14 has NO certifications (missing-field test)
        "project": f"Built a {'prediction model' if profile=='data' else 'web application'} using {', '.join(skills[:3])}.",
    }

def to_lines(c):
    return [
        ("title", c["name"]), ("text", c["email"]),
        ("head", "Skills"), ("text", ", ".join(c["skills"])),
        ("head", "Education"), ("text", c["education"]),
        ("head", "Experience"), ("text", c["experience"]),
        ("head", "Certifications"), ("text", ", ".join(c["certs"]) if c["certs"] else ""),
        ("head", "Projects"), ("text", c["project"]),
    ]

def save_docx(c, path):
    d = docx.Document()
    for kind, txt in to_lines(c):
        if kind == "title": d.add_heading(txt, 0)
        elif kind == "head": d.add_heading(txt, 2)
        else: d.add_paragraph(txt)
    d.save(path)

def save_pdf(c, path):
    st = getSampleStyleSheet()
    story = []
    for kind, txt in to_lines(c):
        style = st["Title"] if kind == "title" else st["Heading2"] if kind == "head" else st["BodyText"]
        story += [Paragraph(txt.replace("&", "&amp;") or " ", style), Spacer(1, 6)]
    SimpleDocTemplate(path).build(story)

for i in range(15):
    c = build_candidate(i)
    base = f"sample_data/resumes/resume_{i+1:02d}_{c['name'].replace(' ', '_')}"
    if i < 10: save_docx(c, base + ".docx")   # first 10 as Word files
    else:      save_pdf(c, base + ".pdf")     # last 5 as PDF files

print(len(os.listdir("sample_data/resumes")), "resumes created")

## 5. Module 1: `extractor.py` (resume extraction)

In [ ]:
%%writefile extractor.py
"""
extractor.py  -  MODULE 1: resume extraction
Reads PDF/DOCX resumes, cleans the text, and pulls out structured fields.
"""
import os
import re
import pdfplumber
import docx

SUPPORTED = (".pdf", ".docx")   # file types we can read


# ---------------------------------------------------------------
# Step 1: read raw text from a file (with clear error messages)
# ---------------------------------------------------------------
def extract_text(path):
    """Read raw text from a PDF or DOCX. Raises clear errors for bad files."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    ext = os.path.splitext(path)[1].lower()
    if ext not in SUPPORTED:
        raise ValueError(f"Unsupported file type '{ext}'. Only PDF and DOCX are allowed.")
    try:
        if ext == ".pdf":
            # pdfplumber reads text page by page
            with pdfplumber.open(path) as pdf:
                text = "\n".join((p.extract_text() or "") for p in pdf.pages)
        else:
            # python-docx reads paragraphs; resumes often hide text in tables too
            d = docx.Document(path)
            parts = [p.text for p in d.paragraphs]
            for t in d.tables:
                for row in t.rows:
                    for cell in row.cells:
                        parts.append(cell.text)
            text = "\n".join(parts)
    except Exception as e:
        raise ValueError(f"Could not read file (corrupted or unreadable): {e}")
    if not text.strip():
        raise ValueError("No text found (file is empty or a scanned image).")
    return text


# ---------------------------------------------------------------
# Step 2: clean and normalize the text
# ---------------------------------------------------------------
def clean_text(text):
    """Normalize text: fix odd characters, extra spaces, keep line breaks."""
    text = text.replace("\u2022", " ").replace("\u2013", "-").replace("\u2014", "-")  # bullets, dashes
    text = re.sub(r"[ \t]+", " ", text)        # collapse spaces/tabs into one space
    text = re.sub(r"\n\s*\n+", "\n", text)     # remove blank lines
    return text.strip()


# ---------------------------------------------------------------
# Step 3: skills dictionary (canonical skill -> list of aliases)
# ---------------------------------------------------------------
SKILLS = {
    # programming
    "python": [], "java": [], "javascript": ["js"], "typescript": [], "c++": [], "c#": [], "php": [], "go": ["golang"],
    # data / ML
    "sql": [], "pandas": [], "numpy": [], "machine learning": ["ml"], "deep learning": ["dl"],
    "scikit-learn": ["sklearn", "scikit learn"], "tensorflow": [], "keras": [], "pytorch": [],
    "nlp": ["natural language processing"], "computer vision": [], "statistics": ["statistical analysis"],
    "data visualization": ["data visualisation"], "data analysis": ["data analytics"],
    "power bi": ["powerbi"], "tableau": [], "excel": ["ms excel"], "matplotlib": [], "seaborn": [],
    "jupyter notebook": ["jupyter"], "big data": [], "spark": ["pyspark"], "hadoop": [],
    # web
    "html": ["html5"], "css": ["css3"], "react": ["reactjs", "react.js"], "angular": [], "vue": ["vue.js"],
    "node.js": ["nodejs", "node js"], "express": ["express.js", "expressjs"], "django": [], "flask": [],
    "rest apis": ["rest api", "restful api", "restful apis", "rest"], "graphql": [], "tailwind": [], "bootstrap": [],
    # databases
    "mongodb": ["mongo"], "mysql": [], "postgresql": ["postgres"], "firebase": [], "redis": [],
    # devops / cloud
    "git": [], "github": [], "docker": [], "kubernetes": ["k8s"], "aws": ["amazon web services"],
    "azure": [], "gcp": ["google cloud"], "ci/cd": ["cicd", "ci cd"], "linux": [], "agile": ["scrum"],
    # soft skills
    "communication": [], "problem solving": ["problem-solving"], "teamwork": ["team work", "collaboration"],
    "leadership": [], "time management": [],
}

# reverse lookup: any alias -> canonical name (so "ML" and "machine learning" are the same skill)
ALIAS_TO_CANON = {}
for _canon, _aliases in SKILLS.items():
    ALIAS_TO_CANON[_canon] = _canon
    for _a in _aliases:
        ALIAS_TO_CANON[_a] = _canon


def extract_skills(text):
    """Dictionary matching with word boundaries. Returns a set of canonical skills."""
    t = text.lower()
    found = set()
    for term, canon in ALIAS_TO_CANON.items():
        # the lookbehind/lookahead stops 'java' from matching inside 'javascript'
        pattern = r"(?<![a-z0-9+#.])" + re.escape(term) + r"(?![a-z0-9+#])"
        if re.search(pattern, t):
            found.add(canon)
    return found


# ---------------------------------------------------------------
# Step 4: split the resume into sections (rule-based)
# ---------------------------------------------------------------
SECTION_HEADS = {
    "skills": ["skills", "technical skills", "key skills"],
    "education": ["education", "academic background", "qualifications"],
    "experience": ["experience", "work experience", "professional experience", "internship", "internships"],
    "certifications": ["certifications", "certificates", "courses"],
    "projects": ["projects", "academic projects", "personal projects"],
}


def split_sections(text):
    """Group lines under the most recent heading they appear after."""
    sections = {k: [] for k in SECTION_HEADS}
    current = None
    for line in text.split("\n"):
        key = line.strip().lower().rstrip(":")
        found = next((s for s, names in SECTION_HEADS.items() if key in names), None)
        if found:
            current = found                      # a heading line: switch section
        elif current:
            sections[current].append(line.strip())   # a normal line: belongs to current section
    return {k: " ".join(v).strip() for k, v in sections.items()}


# ---------------------------------------------------------------
# Step 5: full pipeline for one resume
# ---------------------------------------------------------------
def parse_resume(path):
    """Never crashes: on a bad file it returns a dict with an 'error' message instead."""
    result = {"file": os.path.basename(path), "error": None}
    try:
        raw = extract_text(path)
    except Exception as e:
        result["error"] = str(e)
        return result

    text = clean_text(raw)
    sec = split_sections(text)
    lines = [l for l in text.split("\n") if l.strip()]

    # years of experience: first "<number> years" pattern in the text
    yrs = re.search(r"(\d+)\+?\s*(?:years|yrs|year)", text.lower())
    result.update({
        "name": lines[0] if lines else "Unknown",   # display only - NEVER used in scoring
        "text": text,
        "skills": sorted(extract_skills(text)),
        "education": sec["education"] or "Not found",
        "experience": sec["experience"] or "Not found",
        "certifications": sec["certifications"] or "Not found",
        "projects": sec["projects"] or "Not found",
        "years_experience": int(yrs.group(1)) if yrs else None,
    })
    return result

## 6. Module 2: `matcher.py` (JD keywords + explainable score)

In [ ]:
%%writefile matcher.py
"""
matcher.py  -  MODULE 2: job-description matching and explainable scoring
Final score = 70% skill match + 30% text similarity. No name/age/gender/etc. is ever used.
"""
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from extractor import extract_skills, ALIAS_TO_CANON

# ---------------- scoring settings (documented in README) ----------------
W_REQUIRED, W_OPTIONAL, W_EXTRA = 1.0, 0.5, 0.5   # weight of each kind of keyword
SKILL_PART, SIM_PART = 0.7, 0.3                    # final = 70% skills + 30% text similarity

# words that appear in almost every JD and are not skills
GENERIC = {"looking", "required", "requires", "experience", "candidates", "candidate", "strong", "skills", "skill",
           "knowledge", "good", "expected", "plus", "hiring", "job", "description", "bachelor", "degree", "related",
           "field", "years", "year", "work", "working", "ability", "team", "role", "responsibilities", "preferred",
           "must", "including", "etc", "excellent", "able", "using", "use", "new", "company", "need", "needs", "seeking",
           "proficiency", "understanding", "familiarity", "hands", "science", "computer",
           "scientist", "developer", "stack", "skilled", "full", "engineer", "analyst", "web", "data", "senior",
           "junior", "intern", "tech", "technical"}

# sentences containing these words mark their skills as "optional"
OPTIONAL_CUES = ["plus", "preferred", "nice to have", "good to have", "bonus", "desirable", "advantage"]


def split_sentences(text):
    return [s.strip() for s in re.split(r"\n|(?<=[a-zA-Z0-9])\.\s+", text) if s.strip()]


# ---------------------------------------------------------------
# Step 1: extract keywords from ANY job description
# ---------------------------------------------------------------
def extract_jd_keywords(jd_text, corpus_texts, top_n=5):
    """Layer 1: dictionary skills (required vs optional). Layer 2: TF-IDF phrases for anything new."""
    required, optional = set(), set()
    for sent in split_sentences(jd_text):
        found = extract_skills(sent)
        if any(cue in sent.lower() for cue in OPTIONAL_CUES):
            optional |= found
        else:
            required |= found
    optional -= required          # a skill that is required anywhere stays required

    # TF-IDF: JD vs resumes. Words frequent in the JD but rare in resumes score high.
    covered_words = set(w for term in ALIAS_TO_CANON for w in re.findall(r"[a-z0-9]+", term))
    stops = list(ENGLISH_STOP_WORDS.union(GENERIC))
    vec = TfidfVectorizer(ngram_range=(1, 2), stop_words=stops)
    X = vec.fit_transform([jd_text.lower()] + list(corpus_texts))
    scores = X[0].toarray().ravel()
    terms = vec.get_feature_names_out()

    extras = []
    for idx in scores.argsort()[::-1]:
        if scores[idx] <= 0 or len(extras) >= top_n:
            break
        term = terms[idx]
        if any(w in covered_words for w in term.split()):
            continue                                    # already covered by the dictionary
        if any(term in e or e in term for e in extras):
            continue                                    # skip near-duplicates
        extras.append(term)
    return {"required": sorted(required), "optional": sorted(optional), "extra": extras}


# ---------------------------------------------------------------
# Step 2: score one resume against the JD keywords
# ---------------------------------------------------------------
def anonymize(text):
    """Remove the name (first line) and emails so identity cannot influence text similarity."""
    lines = text.split("\n")[1:]
    return re.sub(r"\S+@\S+", " ", " ".join(lines))


def match_resume(resume, kw, jd_text):
    r_text = resume["text"].lower()
    r_skills = set(resume["skills"])

    items = [(s, W_REQUIRED, "required") for s in kw["required"]] + \
            [(s, W_OPTIONAL, "optional") for s in kw["optional"]] + \
            [(s, W_EXTRA, "extra") for s in kw["extra"]]

    matched, missing, got, total = [], [], 0.0, 0.0
    for term, w, kind in items:
        if kind == "extra":
            hit = bool(re.search(r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])", r_text))
        else:
            hit = term in r_skills
        total += w
        if hit:
            matched.append(term)
            got += w
        else:
            missing.append(term)

    skill_score = 100 * got / total if total else 0.0       # weighted % of keywords found

    # cosine similarity between the JD and the anonymized resume (0-100%)
    vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    M = vec.fit_transform([jd_text.lower(), anonymize(resume["text"]).lower()])
    sim = float(cosine_similarity(M[0], M[1])[0][0]) * 100

    final = SKILL_PART * skill_score + SIM_PART * sim
    n_req = len(kw["required"])
    req_hit = len([m for m in matched if m in kw["required"]])
    reason = (f"Matched {req_hit}/{n_req} required skills; skill score {skill_score:.0f}%, "
              f"text similarity {sim:.0f}% -> final = 0.7*{skill_score:.0f} + 0.3*{sim:.0f} = {final:.1f}")

    return {"file": resume["file"], "name": resume["name"], "final_score": round(final, 1),
            "skill_score": round(skill_score, 1), "similarity": round(sim, 1),
            "matched": matched, "missing": missing, "years": resume["years_experience"], "reason": reason}


# ---------------------------------------------------------------
# Step 3: rank all valid resumes
# ---------------------------------------------------------------
def rank_candidates(parsed, jd_text):
    valid = [p for p in parsed if not p["error"]]      # bad files are skipped, not fatal
    kw = extract_jd_keywords(jd_text, [p["text"] for p in valid])
    df = pd.DataFrame([match_resume(p, kw, jd_text) for p in valid])
    return kw, df.sort_values("final_score", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------
# Sorting options shared by the notebook dashboard and the web app
# ---------------------------------------------------------------
SORT_OPTIONS = {
    "Score: high to low": ("final_score", False),
    "Score: low to high": ("final_score", True),
    "Years of experience: high to low": ("years", False),
    "Name: A to Z": ("name", True),
}


def sort_ranked(ranked, option):
    col, ascending = SORT_OPTIONS[option]
    return ranked.sort_values(col, ascending=ascending, na_position="last", kind="stable")

## 7. Module 3: `app.py` (Streamlit web interface)
Run it in a terminal inside this folder: `streamlit run app.py`

In [ ]:
%%writefile app.py
"""
app.py  -  MODULE 3: Streamlit web interface
Run from a terminal inside this folder:   streamlit run app.py
"""
import os
import glob
import html
import tempfile
import pandas as pd
import streamlit as st
from extractor import parse_resume, extract_text
from matcher import rank_candidates, sort_ranked, SORT_OPTIONS

NOTE = ("Human review required: this score only measures keyword overlap with the job description. "
        "It is NOT a measure of a candidate's ability or suitability. "
        "The final decision always stays with a human recruiter.")

st.set_page_config(page_title="AI Resume Screening", layout="wide")
st.title("AI Resume Screening and Candidate Ranking")
st.warning(NOTE)                       # clear human-review note, always visible


def chips(items, color):
    """Small coloured tags for matched / missing skills."""
    if not items:
        return "<i>none</i>"
    return " ".join(f"<span style='background:{color};color:#000;padding:2px 8px;border-radius:10px;"
                    f"margin:2px;display:inline-block'>{html.escape(s)}</span>" for s in items)


def save_upload(f, folder):
    path = os.path.join(folder, os.path.basename(f.name))
    with open(path, "wb") as out:
        out.write(f.getvalue())
    return path


# ---------------- sidebar: inputs ----------------
with st.sidebar:
    st.header("1. Resumes")
    uploads = st.file_uploader("Upload resumes (PDF or DOCX)", accept_multiple_files=True)
    use_sample = st.checkbox("Also use the sample resumes")

    st.header("2. Job description")
    sample_jds = sorted(glob.glob("sample_data/job_descriptions/*.txt"))
    sample_choice = st.selectbox("Sample JD", ["(none)"] + [os.path.basename(p) for p in sample_jds])
    jd_file = st.file_uploader("Upload JD (txt, pdf, docx)", type=["txt", "pdf", "docx"])
    jd_pasted = st.text_area("...or paste the JD here", height=180)

    st.header("3. Sort")
    sort_option = st.selectbox("Sort candidates by", list(SORT_OPTIONS.keys()))

# ---------------- read the JD (pasted > uploaded > sample) ----------------
tmp = tempfile.mkdtemp()
jd_text = jd_pasted.strip()
if not jd_text and jd_file is not None:
    p = save_upload(jd_file, tmp)
    try:
        jd_text = open(p, encoding="utf-8", errors="ignore").read() if p.lower().endswith(".txt") else extract_text(p)
    except Exception as e:
        st.error(f"Job description error: {e}")
if not jd_text and sample_choice != "(none)":
    jd_text = open(os.path.join("sample_data/job_descriptions", sample_choice)).read()

# ---------------- collect resume files ----------------
paths = [save_upload(f, tmp) for f in uploads]
if use_sample:
    paths += sorted(glob.glob("sample_data/resumes/*"))

if not paths or not jd_text.strip():
    st.info("Add resumes (upload or tick the sample box) and a job description in the sidebar to start.")
    st.stop()

# ---------------- parse + rank ----------------
parsed = [parse_resume(p) for p in paths]
bad = [(p["file"], p["error"]) for p in parsed if p["error"]]
for fname, err in bad:
    st.error(f"Skipped {fname}: {err}")          # clear message for unreadable / unsupported files
if len(bad) == len(parsed):
    st.stop()

kw, ranked = rank_candidates(parsed, jd_text)
if not (kw["required"] or kw["optional"] or kw["extra"]):
    st.warning("No known skills were found in this job description, so all scores will be 0.")

st.subheader("Keywords found in the job description")
c1, c2, c3 = st.columns(3)
c1.markdown("**Required**"); c1.markdown(chips(kw["required"], "#cfe8ff"), unsafe_allow_html=True)
c2.markdown("**Optional**"); c2.markdown(chips(kw["optional"], "#e6e6e6"), unsafe_allow_html=True)
c3.markdown("**Extra (TF-IDF)**"); c3.markdown(chips(kw["extra"], "#f3e5ff"), unsafe_allow_html=True)

# ---------------- ranked table (sortable) ----------------
st.subheader("Ranked candidates")
view = sort_ranked(ranked, sort_option).copy()
table = view[["name", "final_score", "skill_score", "similarity", "years"]].copy()
table["matched"] = view["matched"].apply(len)
table["missing"] = view["missing"].apply(len)
table.insert(0, "rank", range(1, len(table) + 1))
st.dataframe(table, hide_index=True)

# ---------------- candidate detail view ----------------
st.subheader("Candidate detail")
labels = {f"{r['name']}  (score {r['final_score']})": r["file"] for _, r in view.iterrows()}
choice = st.selectbox("Pick a candidate", list(labels.keys()))
row = ranked[ranked["file"] == labels[choice]].iloc[0]
cand = next(p for p in parsed if p["file"] == row["file"])

st.markdown(f"### {row['name']} - final score {row['final_score']}")
st.write("**Why this score:** " + row["reason"])
st.markdown("**Matched skills:** " + chips(row["matched"], "#b7e4c7"), unsafe_allow_html=True)
st.markdown("**Missing skills:** " + chips(row["missing"], "#ffc9c9"), unsafe_allow_html=True)
st.write(f"**Years of experience (detected):** {cand['years_experience']}")
st.write("**Education:** " + cand["education"])
st.write("**Experience:** " + cand["experience"])
st.write("**Certifications:** " + cand["certifications"])
st.write("**Projects:** " + cand["projects"])
st.warning(NOTE)

## 8. Import the modules and test the parser (good files + bad files)

In [ ]:
from extractor import parse_resume, extract_text, extract_skills, clean_text
from matcher import (rank_candidates, match_resume, extract_jd_keywords,
                     sort_ranked, SORT_OPTIONS)

folder = "sample_data/resumes"
parsed = [parse_resume(os.path.join(folder, f)) for f in sorted(os.listdir(folder))]
display(pd.DataFrame([{"file": p["file"], "skills_found": len(p.get("skills", [])),
                       "years": p.get("years_experience"), "certs": p.get("certifications"),
                       "error": p["error"]} for p in parsed]))

# bad files for the test report: wrong type, empty, corrupted, missing
open("sample_data/bad_file.txt", "w").write("hello")
docx.Document().save("sample_data/empty_resume.docx")
open("sample_data/corrupt.pdf", "w").write("not a real pdf")

for bad in ["sample_data/bad_file.txt", "sample_data/empty_resume.docx",
            "sample_data/corrupt.pdf", "sample_data/missing.pdf"]:
    print(bad, "->", parse_resume(bad)["error"])

## 9. Test ranking on both JDs

In [ ]:
for jd_file in sorted(os.listdir("sample_data/job_descriptions")):
    jd_text = open(os.path.join("sample_data/job_descriptions", jd_file)).read()
    kw, ranked = rank_candidates(parsed, jd_text)
    print("=" * 70)
    print("JD:", jd_file)
    print("Required :", kw["required"])
    print("Optional :", kw["optional"])
    print("Extra (TF-IDF):", kw["extra"])
    display(ranked[["name", "final_score", "skill_score", "similarity", "years"]].head(8))
    top = ranked.iloc[0]
    print("Top candidate:", top["name"])
    print("  Matched:", top["matched"])
    print("  Missing:", top["missing"])
    print("  Why    :", top["reason"])

## 10. In-notebook dashboard (upload, rank, sort, candidate detail, human-review note)

In [ ]:
import html as _html

UPLOAD_DIR = "sample_data/uploads_tmp"
os.makedirs(UPLOAD_DIR, exist_ok=True)

NOTE = ("Human review required: this score only measures keyword overlap with the job description. "
        "It is NOT a measure of a candidate's ability or suitability. "
        "The final decision always stays with a human recruiter.")

state = {"parsed": [], "ranked": None, "kw": None, "sample_paths": []}

# ---------- widgets ----------
resume_up  = widgets.FileUpload(accept=".pdf,.docx,.txt", multiple=True, description="Upload resumes")
jd_up      = widgets.FileUpload(accept=".txt,.pdf,.docx", multiple=False, description="Upload JD")
jd_box     = widgets.Textarea(placeholder="Paste job description here...",
                              layout=widgets.Layout(width="100%", height="140px"))
sample_btn = widgets.Button(description="Load sample data", button_style="info")
run_btn    = widgets.Button(description="Rank candidates", button_style="success")
sort_dd    = widgets.Dropdown(options=list(SORT_OPTIONS.keys()), description="Sort by:",
                              layout=widgets.Layout(width="50%"))
cand_dd    = widgets.Dropdown(options=[], description="Candidate:", layout=widgets.Layout(width="70%"))
out_table  = widgets.Output()
out_detail = widgets.Output()

# ---------- helpers ----------
def uploaded_items(w):
    """Works for both ipywidgets 7 (dict) and 8 (tuple)."""
    v = w.value
    if isinstance(v, dict):
        return [(name, bytes(f["content"])) for name, f in v.items()]
    return [(f["name"], bytes(f["content"])) for f in v]

def save_uploads(items):
    paths = []
    for name, content in items:
        p = os.path.join(UPLOAD_DIR, os.path.basename(name))
        with open(p, "wb") as f:
            f.write(content)
        paths.append(p)
    return paths

def get_jd_text():
    items = uploaded_items(jd_up)
    if items:
        p = save_uploads(items[:1])[0]
        if p.lower().endswith(".txt"):
            return open(p, encoding="utf-8", errors="ignore").read()
        return extract_text(p)
    return jd_box.value

def chips(items, color):
    if not items:
        return "<i>none</i>"
    return " ".join(
        f"<span style='background:{color};color:#000;padding:2px 8px;border-radius:10px;"
        f"margin:2px;display:inline-block'>{_html.escape(s)}</span>" for s in items)

def note_box():
    return (f"<div style='background:#fff3cd;color:#000;padding:8px 12px;border-radius:6px;"
            f"border:1px solid #e0c36a'><b>Note:</b> {NOTE}</div>")

# ---------- candidate detail view ----------
def show_detail(change=None):
    with out_detail:
        clear_output()
        if state["ranked"] is None or cand_dd.value is None:
            return
        r = state["ranked"].loc[cand_dd.value]
        p = next(x for x in state["parsed"] if x["file"] == r["file"])
        e = _html.escape
        display(widgets.HTML(f"""
        <div style='line-height:1.7'>
        <h4>{e(str(r['name']))} &nbsp;|&nbsp; Final score: {r['final_score']}</h4>
        <b>Why this score:</b> {e(r['reason'])}<br>
        <b>Matched skills:</b> {chips(r['matched'], '#b7e4c7')}<br>
        <b>Missing skills:</b> {chips(r['missing'], '#ffc9c9')}<br>
        <b>Years of experience (detected):</b> {p['years_experience']}<br>
        <b>Education:</b> {e(p['education'])}<br>
        <b>Experience:</b> {e(p['experience'])}<br>
        <b>Certifications:</b> {e(p['certifications'])}<br>
        <b>Projects:</b> {e(p['projects'])}<br><br>
        {note_box()}
        </div>"""))

# ---------- ranked table (re-drawn whenever the sort option changes) ----------
def render_table(change=None):
    if state["ranked"] is None:
        return
    view = sort_ranked(state["ranked"], sort_dd.value)
    with out_table:
        clear_output()
        display(widgets.HTML(note_box()))
        kw = state["kw"]
        print("Required keywords:", kw["required"])
        print("Optional keywords:", kw["optional"])
        print("Extra keywords   :", kw["extra"])
        for fname, err in state.get("bad", []):
            print(f"Skipped {fname}: {err}")
        table = view[["name", "final_score", "skill_score", "similarity", "years"]].copy()
        table.insert(0, "rank", range(1, len(table) + 1))
        table["matched"] = view["matched"].apply(len)
        table["missing"] = view["missing"].apply(len)
        display(table)
    # dropdown follows the same order; its value is the row index of state["ranked"]
    cand_dd.unobserve(show_detail, names="value")
    cand_dd.options = [(f"{k+1}. {r['name']}  (score {r['final_score']})", i)
                       for k, (i, r) in enumerate(view.iterrows())]
    cand_dd.observe(show_detail, names="value")
    show_detail()

# ---------- button actions ----------
def on_sample(_):
    folder = "sample_data/resumes"
    state["sample_paths"] = [os.path.join(folder, f) for f in sorted(os.listdir(folder))]
    jd_box.value = open("sample_data/job_descriptions/jd_data_scientist.txt").read()
    with out_table:
        clear_output()
        print(f"Loaded {len(state['sample_paths'])} sample resumes + sample JD. Click 'Rank candidates'.")

def on_run(_):
    with out_table:
        clear_output()
        try:
            jd_text = get_jd_text().strip()
        except Exception as e:
            print("Job description error:", e); return
        if not jd_text:
            print("Please paste or upload a job description."); return

        items = uploaded_items(resume_up)
        paths = save_uploads(items) if items else state["sample_paths"]
        if not paths:
            print("Please upload resumes or click 'Load sample data'."); return

        parsed_now = [parse_resume(p) for p in paths]
        bad = [(p["file"], p["error"]) for p in parsed_now if p["error"]]
        if len(bad) == len(parsed_now):
            print("No readable resumes. Errors:", bad); return

        kw, ranked = rank_candidates(parsed_now, jd_text)
        if not (kw["required"] or kw["optional"] or kw["extra"]):
            print("Warning: no known skills found in this JD, so scores will be 0.")
        state.update({"parsed": parsed_now, "ranked": ranked, "kw": kw, "bad": bad})
    render_table()

sample_btn.on_click(on_sample)
run_btn.on_click(on_run)
sort_dd.observe(render_table, names="value")
cand_dd.observe(show_detail, names="value")

display(widgets.VBox([
    widgets.HTML("<h3>AI Resume Screening Dashboard</h3>"),
    widgets.HTML(note_box()),
    widgets.HBox([resume_up, sample_btn]),
    widgets.HTML("<b>Job description</b> (paste below, or upload a file):"),
    jd_box, jd_up, run_btn, sort_dd, out_table,
    widgets.HTML("<hr><b>Candidate detail view</b>"),
    cand_dd, out_detail
]))

## 11. Automated test report (12 tests)

In [ ]:
results = []
def log(test, expected, actual, passed):
    results.append({"test": test, "expected": expected, "actual": actual, "result": "PASS" if passed else "FAIL"})

folder = "sample_data/resumes"
files = sorted(os.listdir(folder))
docx_f = next(f for f in files if f.endswith(".docx"))
pdf_f  = next(f for f in files if f.endswith(".pdf"))

# 1-2. good files (one DOCX, one PDF)
for f in (docx_f, pdf_f):
    p = parse_resume(os.path.join(folder, f))
    log(f"Parse {f.split('.')[-1].upper()} resume", "No error, skills found",
        f"{len(p.get('skills', []))} skills", p["error"] is None and len(p["skills"]) > 0)

# 3-6. bad files must give clear errors
bad_cases = [("Unsupported file type (.txt)", "sample_data/bad_file.txt", "Unsupported"),
             ("Empty DOCX", "sample_data/empty_resume.docx", "No text found"),
             ("Corrupted PDF", "sample_data/corrupt.pdf", "Could not read"),
             ("File not found", "sample_data/missing.pdf", "File not found")]
for name, path, expect in bad_cases:
    err = parse_resume(path)["error"] or ""
    log(name, f"Clear error containing '{expect}'", err[:60], expect in err)

# 7. missing field
p14 = next(p for p in parsed if "Divya" in p["file"])
log("Resume with missing certifications", "'Not found', no crash",
    p14["certifications"], p14["certifications"] == "Not found" and p14["error"] is None)

# 8. JD with no known skills
try:
    kw, r = rank_candidates(parsed, "We need a pastry chef who loves baking bread and cakes.")
    log("JD with no known skills", "No crash, scores near 0", f"top score {r.iloc[0]['final_score']}", True)
except Exception as e:
    log("JD with no known skills", "No crash", str(e), False)

# 9. ranking sanity: the data JD should rank a data-profile resume first
jd_text = open("sample_data/job_descriptions/jd_data_scientist.txt").read()
kw, r = rank_candidates(parsed, jd_text)
top_no = int(re.search(r"resume_(\d+)", r.iloc[0]["file"]).group(1))
log("Data JD ranks data-profile resume first", "Odd resume number (data profile)", f"resume {top_no}", top_no % 2 == 1)

# 10. bias test: change only name + email, the score must not change
orig = parsed[0].copy()
changed = orig.copy()
lines = orig["text"].split("\n")
lines[0], lines[1] = "Zzz Qqq", "zzz.qqq@example.com"
changed["text"] = "\n".join(lines)
changed["skills"] = sorted(extract_skills(changed["text"]))
s1 = match_resume(orig, kw, jd_text)["final_score"]
s2 = match_resume(changed, kw, jd_text)["final_score"]
log("Bias test: change name and email only", "Same score", f"{s1} vs {s2}", s1 == s2)

# 11. one bad file inside a batch must not break the batch
mixed = [parse_resume(p) for p in ["sample_data/bad_file.txt", os.path.join(folder, docx_f)]]
kw2, r2 = rank_candidates(mixed, jd_text)
log("Bad file inside a batch", "Bad file skipped, good file ranked", f"{len(r2)} ranked", len(r2) == 1)

# 12. score consistency: same input twice -> identical output
_, a = rank_candidates(parsed, jd_text)
_, b = rank_candidates(parsed, jd_text)
same = a["final_score"].tolist() == b["final_score"].tolist() and a["file"].tolist() == b["file"].tolist()
log("Score consistency (run twice)", "Identical scores and order", "identical" if same else "different", same)

# 13. sorting works
asc = sort_ranked(a, "Score: low to high")["final_score"].tolist()
log("Sort low to high", "Scores in ascending order", "ok" if asc == sorted(asc) else "wrong", asc == sorted(asc))

report = pd.DataFrame(results)
display(report)
report.to_csv("test_report.csv", index=False)
print(f"{(report.result == 'PASS').sum()}/{len(report)} tests passed. Saved test_report.csv")